<img align="center" src="https://iili.io/3wI8gI.png" style="height:90px" style="width:30px"/>

<h1><center>Churn predictions at Telia</center></h1>

You have been assigned the task of building a model that will predict whether or not a customer of telia will churn. The data for this assignment can be found in the csv file `churn_clf`

<br> 
<div>
<img src="https://strikedeck.com/wp-content/uploads/2016/06/spurnthechurn1.jpg" width="250"/>
</div>
<br> 
<br> 

If the model predicts that a customer will churn, that customer will be sent a special deal to try to keep the customer from leaving telia. If the prediction is correct (a True Positive), the expected gain is 1000 SEK. However, if the prediction is wrong (a False Positive), the expected loss is 1500 SEK. 

Your goal is to build the most profitable model possible.

Guidelines:
- Use a LogisticRegression model
- Test size = 0.2
- You are not allowed to change the model after looking at the performance on test data
- Your models predictions on test data will be translated into SEK. I.e:
    - 10 TP = 10 * 1000 SEK = + 10 000 
    - 10 FP = 10 * -1500 SEK = - 15 000 SEK
        - Expected Value from model = - 5000 SEK
        
<hr style="border:3px solid pink"> </hr>


In [31]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn
import numpy as np

df = pd.read_csv("churn_clf.csv")

In [32]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7010-BRBUU,Male,No,Yes,Yes,72,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic),24.10,1734.65,No
1,9688-YGXVR,Female,No,No,No,44,Yes,No,Fiber optic,No,...,Yes,No,Yes,No,Month-to-month,Yes,Credit card (automatic),88.15,3973.20,No
2,9286-DOJGF,Female,Yes,Yes,No,38,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),74.95,2869.85,Yes
3,6994-KERXL,Male,No,No,No,4,Yes,No,DSL,No,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,55.90,238.50,No
4,2181-UAESM,Male,No,No,No,2,Yes,No,DSL,Yes,...,Yes,No,No,No,Month-to-month,No,Electronic check,53.45,119.50,No


In [33]:
df.drop(columns="customerID",inplace=True)

In [34]:

yes_no_list = ["SeniorCitizen","Partner","Dependents","PhoneService","MultipleLines","OnlineSecurity","OnlineBackup"
               ,"DeviceProtection","TechSupport","StreamingTV","StreamingMovies","PaperlessBilling","Churn"]


In [35]:
scale = {"Yes":1,"No":0,"No internet service": 0,"No phone service":0}

yes_no_list = ["SeniorCitizen","Partner","Dependents","PhoneService","MultipleLines","OnlineSecurity","OnlineBackup"
               ,"DeviceProtection","TechSupport","StreamingTV","StreamingMovies","PaperlessBilling","Churn"]
for i in yes_no_list:
    df[i] = df[i].map(scale)




In [36]:
df.head(10)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Male,0,1,1,72,1,1,No,0,0,0,0,0,0,Two year,0,Credit card (automatic),24.10,1734.65,0
1,Female,0,0,0,44,1,0,Fiber optic,0,1,1,0,1,0,Month-to-month,1,Credit card (automatic),88.15,3973.20,0
2,Female,1,1,0,38,1,1,Fiber optic,0,0,0,0,0,0,Month-to-month,1,Bank transfer (automatic),74.95,2869.85,1
3,Male,0,0,0,4,1,0,DSL,0,0,0,0,0,1,Month-to-month,1,Electronic check,55.90,238.50,0
4,Male,0,0,0,2,1,0,DSL,1,0,1,0,0,0,Month-to-month,0,Electronic check,53.45,119.50,0
5,Female,0,1,0,70,0,0,DSL,1,0,1,1,0,1,Two year,1,Bank transfer (automatic),49.85,3370.20,0
6,Female,0,0,0,33,1,1,Fiber optic,1,0,0,0,0,1,Month-to-month,1,Electronic check,90.65,2989.60,0
7,Female,0,0,0,1,0,0,DSL,0,0,0,0,0,0,Month-to-month,1,Mailed check,24.90,24.90,0
8,Male,0,0,0,39,0,0,DSL,0,0,1,1,0,0,One year,0,Mailed check,35.55,1309.15,0
9,Male,1,0,0,55,1,1,Fiber optic,1,1,1,1,1,1,Month-to-month,1,Electronic check,116.50,6382.55,0


In [37]:
for c in yes_no_list:
    if len(df[c].value_counts()) > 2:
        print(c, df[c].unique())

In [38]:
df_copy = df

In [39]:
scale_gender = {"Female":0, "Male":1}
df["is_male"] = df["gender"].map(scale_gender)
df.drop(columns="gender",inplace=True)

In [40]:
df_copy

,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,is_male
0,0,1,1,72,1,1,No,0,0,0,0,0,0,Two year,0,Credit card (automatic),24.10,1734.65,0,1
1,0,0,0,44,1,0,Fiber optic,0,1,1,0,1,0,Month-to-month,1,Credit card (automatic),88.15,3973.20,0,0
2,1,1,0,38,1,1,Fiber optic,0,0,0,0,0,0,Month-to-month,1,Bank transfer (automatic),74.95,2869.85,1,0
3,0,0,0,4,1,0,DSL,0,0,0,0,0,1,Month-to-month,1,Electronic check,55.90,238.50,0,1
4,0,0,0,2,1,0,DSL,1,0,1,0,0,0,Month-to-month,0,Electronic check,53.45,119.50,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5981,0,1,0,1,1,0,Fiber optic,1,0,0,0,1,1,Month-to-month,1,Electronic check,95.00,95.00,1,1
5982,0,1,1,23,1,1,DSL,1,1,1,1,1,1,Two year,1,Credit card (automatic),91.10,2198.30,0,0
5983,0,1,1,12,1,0,No,0,0,0,0,0,0,Month-to-month,1,Electronic check,21.15,306.05,0,1
5984,1,0,0,12,1,1,Fiber optic,0,0,1,0,1,1,Month-to-month,1,Electronic check,99.45,1200.15,1,1


In [41]:
for i in df.columns:
    if len(df[i].value_counts()) > 2:
        print(i, df[i].unique())

tenure [72 44 38  4  2 70 33  1 39 55 52 30 60 50 32 51 69 42 14 62  5 63 67 40
 65 16 46 11 49 68 10 53 54 15  3 71  8 64 57 20 26 31  7 35  6 13 23  9
 45 17 34 58 12 25 28 29 43 19 41 37 27 22 24 18 56 66 59 48 47 61 21  0
 36]
InternetService <StringArray>
['No', 'Fiber optic', 'DSL']
Length: 3, dtype: str
Contract <StringArray>
['Two year', 'Month-to-month', 'One year']
Length: 3, dtype: str
PaymentMethod <StringArray>
[  'Credit card (automatic)', 'Bank transfer (automatic)',
          'Electronic check',              'Mailed check']
Length: 4, dtype: str
MonthlyCharges [24.1  88.15 74.95 ... 59.25 35.35 21.15]
TotalCharges [1734.65 3973.2  2869.85 ...  306.05 1200.15  457.3 ]


In [42]:
df = pd.get_dummies(df, columns=["InternetService", "Contract", "PaymentMethod"])

In [43]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df.isna().sum() > 0

SeniorCitizen                              False
Partner                                    False
Dependents                                 False
tenure                                     False
PhoneService                               False
MultipleLines                              False
OnlineSecurity                             False
OnlineBackup                               False
DeviceProtection                           False
TechSupport                                False
StreamingTV                                False
StreamingMovies                            False
PaperlessBilling                           False
MonthlyCharges                             False
TotalCharges                               False
Churn                                      False
is_male                                    False
InternetService_DSL                        False
InternetService_Fiber optic                False
InternetService_No                         False
Contract_Month-to-mo

In [44]:
d = df
d["avg_charge"] = d["TotalCharges"] / (d["tenure"] + 1)
d["pay_gap"] = d["MonthlyCharges"] - d["avg_charge"]
d["additions"] = d[["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                    "TechSupport", "StreamingTV", "StreamingMovies"]].sum(axis=1)
d["protection"] = d[["OnlineSecurity", "TechSupport", "OnlineBackup", "DeviceProtection"]].sum(axis=1)
d["streaming"] = d[["StreamingTV", "StreamingMovies"]].sum(axis=1)
d["family"] = d["Partner"] + d["Dependents"]
d["new_cust"] = (d["tenure"] <= 6).astype(int)
d["fiber"] = d["InternetService_Fiber optic"]
d["m2m"] = d["Contract_Month-to-month"]
d["echeck"] = d["PaymentMethod_Electronic check"]
d["m2m_x_tenure"] = d["m2m"] * d["tenure"]
d["m2m_x_fiber"] = d["m2m"] * d["fiber"]
d["m2m_x_echeck"] = d["m2m"] * d["echeck"]
d["fiber_x_charge"] = d["fiber"] * d["MonthlyCharges"]
d["senior_fiber"] = d["SeniorCitizen"] * d["fiber"]
d["m2m_new"] = d["m2m"] * d["new_cust"]
for k in [12, 24, 48]:
    d[f"t_gt{k}"] = (d["tenure"] > k).astype(int)

df = d



In [45]:

X = df.drop(columns="Churn")

y = df["Churn"]

X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, test_size = 0.2,random_state=42)

In [46]:
scaler = sklearn.preprocessing.StandardScaler()

clf = sklearn.linear_model.LogisticRegression()

pipe = sklearn.pipeline.make_pipeline(scaler, clf)

pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](45,)","['SeniorCitizen','Partner','Dependents',...,'t_gt12','t_gt24','t_gt48']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,45
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [ ]:
# DO NOT USE :)))))))
# pipe.score(X_train, y_train)

0.8049289891395155

In [ ]:
y_pred = pipe.predict(X_test)


In [50]:
y_pred

array([0, 1, 0, ..., 1, 0, 1], shape=(1198,))

In [51]:
sklearn.metrics.accuracy_score(y_test, y_pred)

0.8063439065108514